# 2026-05-27 从 `step_san_gap_analysis` 反推的 Prompt / Schema 改进清单

这份 notebook 是**直接根据** `notebooks/03-ssm-design/2026-05-27_step_san_gap_analysis.ipynb` 反推出来的后续改进清单。

命名上明确保留了 `from_step_san_gap_analysis`，用于说明这份记录不是独立分析，而是以 gap analysis 里的问题为输入，进一步反推：

- prompt 该怎么改
- schema / generation contract 该怎么改
- 代码里已经实际做了哪些对应改进


## 1. 从 gap analysis 提炼出的核心问题

根据 `2026-05-27_step_san_gap_analysis.ipynb`，当前生成结果相对参考 san 文件的主要差距有：

1. 没输出完整 `<template> / <script> / <style>` 结构
2. 没显式输出 `const san = require('san')`、`DataTypes`、`module.exports`
3. `dataTypes` 用了 `'number'`，没有对齐 `DataTypes.number`
4. `initData()` 和 `inited()` 的初始化职责没有拆开
5. 没显式输出 `name`
6. 样式没有输出
7. 方法组织形式残留 `methods: {}` 的 Vue 风格
8. 模板细节格式化不够接近参考实现


## 2. 反推后的 prompt 改进清单

针对上述问题，prompt 层需要从“生成可运行代码”升级为“生成可直接落盘、结构和风格接近项目参考 san 的完整文件”。

建议清单如下：

- 明确要求输出完整 `.san` 单文件组件，而不是只输出组件对象片段
- 明确要求必须包含 `<template>`、`<script>`、`<style>` 三段
- 明确要求 `<script>` 中必须包含 `const san = require('san')`、`const DataTypes = san.DataTypes`、`module.exports = san.defineComponent(...)`
- 明确要求必须输出 `name` 字段
- 明确要求 `dataTypes` 优先映射为 `DataTypes.number / string / bool / array / object`，而不是普通字符串
- 明确要求 `initData()` 只负责稳定默认值，prop -> state 同步优先放在 `inited/attached`
- 明确要求样式块不能省略，要尽量复用 `styles` / `style_model` 中已有 CSS
- 明确要求方法尽量定义在组件对象顶层，不保留 Vue 风格 `methods: {}` 包裹
- 明确要求不要输出 Markdown 代码围栏和解释性文字


In [ ]:
# local_server/api/evaluation_routes.py 中更新后的 prompt 关键片段
def _build_generation_prompt(ssm: dict[str, Any], extra_instruction: str = "") -> str:
    ssm_json = json.dumps(ssm, ensure_ascii=False, indent=2)
    return (
        "请仅基于给定 SSM 生成可运行、可直接落盘的 San 单文件组件代码。\n"
        "输出要求：\n"
        "1. 直接输出完整 .san 文件内容，必须包含 <template>、<script>、<style> 三段，不要只输出组件对象片段。\n"
        "2. <script> 中必须显式包含 const san = require('san')、const DataTypes = san.DataTypes，并使用 module.exports = san.defineComponent(...) 导出。\n"
        "3. 组件中必须显式输出 name，值来自 metadata.component_name。\n"
        "4. props 要映射到 DataTypes.number / string / bool / array / object 等标准 San 类型，不要写成普通字符串 'number'。\n"
        "5. data 初始化优先拆成稳定默认值 + 生命周期同步：initData 负责默认值，props 到 state 的同步优先放到 inited/attached 中。\n"
        "6. 优先将 methods 直接定义在组件对象顶层，不要保留 Vue 风格的 methods: {} 包裹，除非 SSM 明确要求。\n"
        "7. 必须尽量完整输出样式，复用 styles 和 style_model 中已有的 class、selector、css_rules，不要省略 <style>。\n"
        "8. 模板中保留现有 DOM 结构、类名、文本内容与事件绑定；插值格式尽量使用 {{ value }} 这种更易读风格。\n"
        "9. 若 SSM 信息不足，可做最小必要假设，但不要编造未出现的复杂业务逻辑。\n"
        "10. 除代码外不要输出解释、标题或 Markdown 代码围栏。\n\n"
        f"补充要求：{extra_instruction or '保持输出风格尽量贴近项目内已有 san 组件：完整 SFC、显式 DataTypes、保留样式、代码可直接保存为 .san 文件。'}\n\n"
        "--- SSM ---\n"
        f"{ssm_json}"
    )


### 解释

这次 prompt 调整的核心思想是：

- 从“让模型理解迁移语义”变成“让模型直接对齐项目中的理想 san 文件形态”
- 把之前 gap analysis 里暴露的缺口，直接转成强约束的输出要求


## 3. 反推后的 schema / generation contract 改进清单

仅改 prompt 还不够，因为 prompt 依赖的是 `SSM` 里已有的结构化约束。

所以还需要同步增强 `san_generation_contract`，让模型从 SSM 本身就能读到更明确的生成规范。

建议清单如下：

- 在 `san_syntax_requirements` 中明确要求输出完整 `.san` SFC 结构
- 明确要求脚本块必须有 `require('san')`、`DataTypes`、`module.exports`
- 明确要求 `name` 来自 `metadata.component_name`
- 明确要求 `props` 映射成标准 `DataTypes.*`，而不是字符串类型
- 明确要求 `initData()` 只承载稳定默认值
- 明确要求“来源于 props 的 data 字段”优先在 `inited/attached` 中同步
- 明确要求“方法优先写在组件对象顶层”
- 明确要求“样式块需要输出，不能只保留类名而丢掉 CSS”


In [ ]:
# SSM/extractors/factory.py 中更新后的 generation contract 关键片段
"san_syntax_requirements": [
    "输出完整的 .san 单文件组件，包含 <template>、<script>、<style> 三个代码块；若无样式也保留空 style 或显式说明无样式",
    "脚本块显式引入 san：const san = require('san')，并使用 module.exports = san.defineComponent(...) 导出组件",
    "props 使用 san.DataTypes 声明，与 script.options.props 一一对应；优先输出 DataTypes.number / string / bool / array / object",
    "组件定义使用 san.defineComponent",
    "组件名来自 metadata.component_name，并显式输出 name 字段",
    "data 使用 initData 返回默认值，与 script.options.data 一一对应；不要在 initData 中直接依赖复杂运行时读取",
    "当 data 字段来源于 props 或外部初始化时，优先在 inited/attached 中用 this.data.set 完成同步",
    "模板中使用 s-if、s-for、on-event、value={= =}、checked={= =} 等 San 语法",
    "状态访问统一使用 this.data.get() / this.data.set()",
    "优先将 methods 直接定义在组件对象顶层，避免保留 Vue 风格的 methods: {} 包裹",
    "子组件标签使用短横线命名",
    "子组件在 components 中显式注册",
    "样式块需尽量完整输出，优先复用 styles/style_model 中已有 class 与 css_rules 信息",
    "定时器、timeout、外部监听等副作用在 attached/disposed 中管理",
]


### 解释

这一步是把 gap analysis 里的“参考 san 文件有哪些结构特征”变成机器可读的 contract。

好处是：

- 不只依赖自然语言 prompt
- SSM 自己也能向模型传递更清晰的最终代码组织要求


## 4. 已完成的实际改进

基于这份 gap analysis，当前已经落地的改进包括：

- 已增强 `local_server/api/evaluation_routes.py` 的 prompt
- 已增强 `SSM/extractors/factory.py` 的 `san_generation_contract`
- 已保留自动去 Markdown 代码围栏逻辑，避免保存 `.san` 时混入 ```javascript 包装
- 已保留“只保存一份输出文件”的逻辑，避免默认路径与自定义路径同时残留


## 5. 新的 `step.san` 生成效果

在应用这轮 prompt / schema 改进后，重新生成了 `data/experiments/step.san`。

这次结果相较于早期版本已经明显改善：

- 已输出完整 `<template>` / `<script>` / `<style>` 三段
- 已补齐 `const san = require('san')`
- 已补齐 `const DataTypes = san.DataTypes`
- 已补齐 `module.exports = san.defineComponent(...)`
- 已显式输出 `name: 'StepCard'`
- 已补齐完整样式块
- 已拆成 `initData()` + `inited()` 两段式初始化
- 已去掉 Markdown 代码围栏


In [ ]:
# 当前新的 data/experiments/step.san 源代码
<template>
    <div class="step-card" on-click="addSteps">
        <div class="steps-display">
            <span class="steps-value">{{steps}}</span>
            <span class="steps-unit">步</span>
        </div>
        <div class="steps-label">今日步数</div>
        <div class="hint">点击卡片 +1000 步</div>
    </div>
</template>

<script>
const san = require('san');
const DataTypes = san.DataTypes;

module.exports = san.defineComponent({
    name: 'StepCard',

    propTypes: {
        initialSteps: DataTypes.number
    },

    initData() {
        return {
            steps: 0
        };
    },

    inited() {
        const initialSteps = this.data.get('initialSteps');
        if (initialSteps != null && !isNaN(initialSteps)) {
            this.data.set('steps', initialSteps);
        }
    },

    addSteps() {
        const current = this.data.get('steps') || 0;
        this.data.set('steps', current + 1000);
    }
});
</script>

<style scoped>
.step-card {
  width: 260px;
  padding: 24px 20px;
  background: linear-gradient(135deg, #43c6ac 0%, #191654 100%);
  border-radius: 20px;
  text-align: center;
  color: white;
  cursor: pointer;
  transition: transform 0.2s;
  font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
}
.step-card:hover {
  transform: translateY(-3px);
  box-shadow: 0 8px 20px rgba(0, 0, 0, 0.2);
}
.steps-display {
  margin-bottom: 12px;
}
.steps-value {
  font-size: 48px;
  font-weight: bold;
}
.steps-unit {
  font-size: 20px;
  margin-left: 4px;
  opacity: 0.8;
}
.steps-label {
  font-size: 18px;
  margin-bottom: 8px;
  opacity: 0.9;
}
.hint {
  font-size: 12px;
  opacity: 0.6;
}
</style>


## 6. 这次新结果还剩的问题

虽然这次生成结果已经非常接近参考 san 文件，但仍然还有几个残留问题：

### 6.1 `propTypes` 应为 `dataTypes`
- 当前输出：`propTypes`
- 参考实现：`dataTypes`
- 这是当前最关键的剩余问题，因为它直接影响和项目内参考组件的一致性

### 6.2 `initData()` 里没有保留 `initialSteps: 0`
- 参考实现会同时初始化 `initialSteps: 0` 和 `steps: 0`
- 当前结果只初始化了 `steps: 0`
- 功能上未必立刻出错，但少了一层更稳妥的默认值保留

### 6.3 模板插值格式仍未对齐
- 当前：`{{steps}}`
- 参考：`{{ steps }}`
- 这属于格式一致性问题，不影响功能

### 6.4 `<style scoped>` 与参考 `<style>` 不完全一致
- 当前结果保留了 `scoped`
- 参考文件没有 `scoped`
- 这未必是 bug，但说明模型仍然更偏向照搬来源特征，而不是严格贴齐参考 san 成品


## 7. 对后续改进的进一步反推

这次新结果说明 prompt / schema 改进已经有效，但要继续逼近参考 san 文件，还可以进一步补三类约束：

- 明确要求使用 `dataTypes`，禁止输出 `propTypes`
- 明确要求 props 自身默认值也要在 `initData()` 中保留一份稳定初始值
- 明确要求模板格式尽量贴近参考实现，包括 `{{ value }}` 这种空格风格


## 8. 还可以继续加强的方向

虽然这次已经把主要 gap 反推成 prompt / schema 约束，但还可以继续做两类增强：

### 8.1 schema 结构增强
- 给 `script.options.data` 增加更结构化的“初始化来源”字段
- 明确某个 data 字段是否来自 prop、默认值是什么、推荐同步生命周期是什么
- 给 `styles` 增加更直接可拼接成 `<style>` 的输出视图

### 8.2 生成后校验增强
- 校验输出中是否真的包含 `<template>` / `<script>` / `<style>`
- 校验是否包含 `require('san')` / `module.exports` / `DataTypes`
- 校验是否真正落了样式而不是只保留类名
- 校验是否还残留 `methods: {}` 这类 Vue 风格结构
- 校验是否错误输出了 `propTypes` 而不是 `dataTypes`


## 9. 一句话结论

这份改进清单的核心价值在于：把 `step_san_gap_analysis` 中“生成代码离参考实现差在哪里”的结论，直接落成了可执行的 prompt 约束和 schema contract，并且经过新的 `step.san` 结果验证，已经显著缩小了与参考 san 文件的差距；当前最值得优先继续追踪的问题是 `propTypes -> dataTypes`。
